In [ ]:
pip install catboost xgboost scikit-learn==1.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 20.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 53.4 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
import gensim
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import torch
import math
from sklearn.cluster import DBSCAN
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, RobertaModel, RobertaTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
leetcode_questions_df = pd.read_csv('/content/drive/MyDrive/thesis/leetcode/part4 feature-engineering/leetcode_questions_df.csv')

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61834 entries, 0 to 61833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            61834 non-null  object 
 1   country                             61834 non-null  object 
 2   contest_url                         61834 non-null  object 
 3   num_of_contest                      61834 non-null  int64  
 4   is_weekly                           61834 non-null  bool   
 5   rank                                61834 non-null  int64  
 6   score                               61834 non-null  int64  
 7   question_number                     61834 non-null  int64  
 8   question_language                   61834 non-null  object 
 9   question_code                       61834 non-null  object 
 10  number_of_lines                     61834 non-null  int64  
 11  names_set                           61834

In [ ]:
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_language'] == 'python']
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_number'] > 2]
leetcode_questions_df = leetcode_questions_df.drop_duplicates(subset=['question_code'])
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['country'].isin(leetcode_questions_df['country'].value_counts().head(2).index)]

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3661 entries, 193 to 61825
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            3661 non-null   object 
 1   country                             3661 non-null   object 
 2   contest_url                         3661 non-null   object 
 3   num_of_contest                      3661 non-null   int64  
 4   is_weekly                           3661 non-null   bool   
 5   rank                                3661 non-null   int64  
 6   score                               3661 non-null   int64  
 7   question_number                     3661 non-null   int64  
 8   question_language                   3661 non-null   object 
 9   question_code                       3661 non-null   object 
 10  number_of_lines                     3661 non-null   int64  
 11  names_set                           3661 non-

In [ ]:
model_name = "neulab/codebert-python"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name)

code_snippets = leetcode_questions_df.question_code.tolist()

# Step 1: Encode the code snippets using CodeBERT
def get_embeddings(code_snippet):
    inputs = tokenizer(code_snippet, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the last hidden state of the [CLS] token as the embedding
    return outputs.last_hidden_state[:, 0, :].numpy()

# Get embeddings for all code snippets
embeddings = np.vstack([get_embeddings(snippet) for snippet in code_snippets])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/703 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at neulab/codebert-python and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
min_samples = 10 ** (math.floor(math.log10(len(code_snippets))) - 1)

min_samples

100

In [ ]:
# Step 2: Apply DBSCAN for clustering and outlier detection
dbscan = DBSCAN(eps=0.1, min_samples=min_samples, metric='cosine', n_jobs=-1)
db_labels = dbscan.fit_predict(embeddings)

# Step 3: Identify and handle outliers
outliers = np.where(db_labels == -1)[0]

# Output some statistics
print(f'Removed {len(outliers)} outliers.')
print(f'Retained {len(db_labels) - len(outliers)} code snippets.')

Removed 6 outliers.
Retained 3655 code snippets.


In [ ]:
# Remove outliers from the DataFrame
leetcode_questions_df.reset_index(drop=True, inplace=True)
leetcode_questions_df = leetcode_questions_df[~leetcode_questions_df.index.isin(outliers)]

leetcode_questions_df.country.value_counts()

,count
country,
United States,2382
India,1273


In [ ]:
leetcode_questions_df.drop(['num_of_contest','is_weekly','rank','score','global_rank_percentile','num_contests_participated','question_number'],axis=1, inplace=True)

<ipython-input-11-968ae4088e84>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leetcode_questions_df.drop(['num_of_contest','is_weekly','rank','score','global_rank_percentile','num_contests_participated','question_number'],axis=1, inplace=True)


In [ ]:
X=leetcode_questions_df.drop('country',axis=1)
Y=leetcode_questions_df.country

In [ ]:
X_nontext=X.select_dtypes(exclude=['object'])
X_nontext.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3655 entries, 0 to 3660
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   number_of_lines                     3655 non-null   int64  
 1   token_count                         3655 non-null   int64  
 2   variables_count                     3655 non-null   int64  
 3   function_count                      3655 non-null   int64  
 4   loop_count                          3655 non-null   int64  
 5   condition_count                     3655 non-null   int64  
 6   single_line_comment_density         3655 non-null   float64
 7   multiline_comment_density           3655 non-null   float64
 8   function_density                    3655 non-null   float64
 9   loop_density                        3655 non-null   float64
 10  condition_density                   3655 non-null   float64
 11  comment_tokens_density              3655 non-nul

In [ ]:
X_train_nontext, X_test_nontext, y_train, y_test = train_test_split(X_nontext, Y, test_size=0.2, random_state=0,stratify=Y)

In [ ]:
X_train_text, X_test_text, Y_train, y_test = train_test_split(X.question_code, Y, test_size=0.2, random_state=0,stratify=Y)

In [ ]:
def read_corpus(text, tokens_only=False):
    for i, line in enumerate(text):
        tokens = gensim.utils.simple_preprocess(line)
        if tokens_only:
            yield tokens
        else:
        # For training data, add tags
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

train_corpus = list(read_corpus(X_train_text))
test_corpus = list(read_corpus(X_test_text, tokens_only=True))

In [ ]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=300, min_count=2)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=55)

In [ ]:
vectors = [model.infer_vector(train_corpus[doc_id].words) for doc_id in range(len(train_corpus))]
X_train_doc2vec = np.vstack(vectors)

test_vectors = [model.infer_vector(test_corpus[doc_id]) for doc_id in range(len(test_corpus))]
X_test_doc2vec = np.vstack(test_vectors)

X_train_doc2vec.shape , X_test_doc2vec.shape

((2924, 300), (731, 300))

In [ ]:
X_train_combined=pd.concat([pd.DataFrame(X_train_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_train_nontext.index),X_train_nontext],axis=1)

X_test_combined=pd.concat([pd.DataFrame(X_test_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_test_nontext.index),X_test_nontext],axis=1)

# Cat Boost

In [ ]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',CatBoostClassifier(iterations=10))])

In [ ]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

Learning rate set to 0.5
0:	learn: 0.6442213	total: 48.3ms	remaining: 435ms
1:	learn: 0.6201293	total: 49.5ms	remaining: 198ms
2:	learn: 0.6061808	total: 50.6ms	remaining: 118ms
3:	learn: 0.5944270	total: 51.8ms	remaining: 77.6ms
4:	learn: 0.5826976	total: 52.9ms	remaining: 52.9ms
5:	learn: 0.5745356	total: 54.1ms	remaining: 36.1ms
6:	learn: 0.5679148	total: 55.2ms	remaining: 23.7ms
7:	learn: 0.5608844	total: 56.4ms	remaining: 14.1ms
8:	learn: 0.5568404	total: 57.5ms	remaining: 6.38ms
9:	learn: 0.5549078	total: 58.6ms	remaining: 0us
Learning rate set to 0.5
0:	learn: 0.6458855	total: 1.39ms	remaining: 12.5ms
1:	learn: 0.6183315	total: 2.46ms	remaining: 9.84ms
2:	learn: 0.6087983	total: 3.59ms	remaining: 8.38ms
3:	learn: 0.5970170	total: 4.74ms	remaining: 7.12ms
4:	learn: 0.5900429	total: 5.96ms	remaining: 5.96ms
5:	learn: 0.5820426	total: 7.12ms	remaining: 4.74ms
6:	learn: 0.5762586	total: 8.33ms	remaining: 3.57ms
7:	learn: 0.5695702	total: 9.48ms	remaining: 2.37ms
8:	learn: 0.5659987	

In [ ]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6606897	total: 1.68ms	remaining: 15.1ms
1:	learn: 0.6360382	total: 3.12ms	remaining: 12.5ms
2:	learn: 0.6209979	total: 4.36ms	remaining: 10.2ms
3:	learn: 0.6120095	total: 5.58ms	remaining: 8.37ms
4:	learn: 0.6018636	total: 6.85ms	remaining: 6.85ms
5:	learn: 0.5923710	total: 8.1ms	remaining: 5.4ms
6:	learn: 0.5895412	total: 9.29ms	remaining: 3.98ms
7:	learn: 0.5846797	total: 10.5ms	remaining: 2.63ms
8:	learn: 0.5812244	total: 11.8ms	remaining: 1.31ms
9:	learn: 0.5785537	total: 13ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.3}
Best F1 score found:  0.5412279986457308
Test set F1 score:  0.5219570785035577


# Cat with Text

In [ ]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6570368	total: 19ms	remaining: 171ms
1:	learn: 0.6299435	total: 33.5ms	remaining: 134ms
2:	learn: 0.6025018	total: 47.9ms	remaining: 112ms
3:	learn: 0.5871050	total: 62ms	remaining: 93ms
4:	learn: 0.5754498	total: 76.1ms	remaining: 76.1ms
5:	learn: 0.5632538	total: 89.9ms	remaining: 59.9ms
6:	learn: 0.5497518	total: 104ms	remaining: 44.6ms
7:	learn: 0.5384503	total: 119ms	remaining: 29.7ms
8:	learn: 0.5280178	total: 133ms	remaining: 14.7ms
9:	learn: 0.5163943	total: 147ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 5, 'classifier__learning_rate': 0.3}
Best F1 score found:  0.5656188895831987
Test set F1 score:  0.5217603199618589


# Support Vector Classifier

In [ ]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',SVC(probability=True))])

In [ ]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6740791476407914


In [26]:
param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto'],
    'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 32 candidates, totalling 160 fits


KeyboardInterrupt: 

# SVC with Text

In [27]:
param_grid = {
    'classifier__C': [0.01 ,0.1, 1],
    'classifier__gamma': ['scale'],
    'classifier__kernel': ['rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters found:  {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Best F1 score found:  0.6511171921455213
Test set F1 score:  0.5975151413592512


# XGBoost

In [28]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',XGBClassifier())])

In [29]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [30]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6583479686219412


In [31]:
param_grid = {
    'n_estimators': [25, 50, 100],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.4, 0.6, 0.8],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss')

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best F1 score found:  0.6076462135657398
Test set F1 score:  0.5967604433077579


# XGBoost With Text

In [32]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss', random_state=0)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.6}
Best F1 score found:  0.6567669751418709
Test set F1 score:  0.6632358013802172


In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.42      0.52       255
           1       0.74      0.89      0.81       476

    accuracy                           0.73       731
   macro avg       0.71      0.66      0.66       731
weighted avg       0.72      0.73      0.71       731

